## Import Libaries

In [37]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import root_mean_squared_error,f1_score,accuracy_score 


## Import Dataset

In [38]:
df=pd.read_csv("Titanic-Dataset.csv")
df=df.drop(columns=['PassengerId','Name','Ticket','Fare','Cabin'])
# Checked for missing values
df.isnull().sum()
# found 177 in the age columndf
# lets use Group-based median imputation to fix it
df['Age']=df['Age'].fillna(df.groupby("Pclass")["Age"].transform("median"))
df.isnull().sum()
#checked again for mossing values
#found 2 missing values in the embarked column, so i will the 2 rows
df=df.dropna()
x = df.iloc[:,1:]
y = df.iloc[:,0].values
df

,Survived,Pclass,Sex,Age,SibSp,Parch,Embarked
0,0,3,male,22.0,1,0,S
1,1,1,female,38.0,1,0,C
2,1,3,female,26.0,0,0,S
3,1,1,female,35.0,1,0,S
4,0,3,male,35.0,0,0,S
...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,S
887,1,1,female,19.0,0,0,S
888,0,3,female,24.0,1,2,S
889,1,1,male,26.0,0,0,C


In [39]:
cat_cols = x.select_dtypes(include="object").columns.tolist()
num_cols = x.select_dtypes(exclude="object").columns.tolist()
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols),
])

## Spitting Training and Test set

In [40]:
x_train,x_test,y_train,y_test=train_test_split(x,y,random_state=42,test_size=0.2)

In [41]:
log_model=Pipeline([
    ("perp",preprocess),
    ("log",LogisticRegression())
])
log_model.fit(x_train,y_train)
log_pred=log_model.predict(x_test)

In [42]:
svm=Pipeline([
    ("perp",preprocess),
    ("svm",SVC(kernel='rbf'))
]
)
svm.fit(x_train,y_train)
svm_pred=svm.predict(x_test)

In [43]:
dt_pipline =Pipeline([("prep",preprocess),("dt",DecisionTreeClassifier())])
param_grid ={"dt__max_depth":[None,10,15,20,30,25]}
dt_search=GridSearchCV(dt_pipline,param_grid,cv=5,scoring="accuracy",n_jobs=-1)
dt_search.fit(x_train,y_train)
dt_pred=dt_search.predict(x_test)

In [44]:
rf_pipline=Pipeline([("prep",preprocess),("rf",RandomForestClassifier())])
param_grid={"rf__n_estimators":[100,150,200,350,300]}
rf_search=GridSearchCV(rf_pipline,param_grid,cv=5,scoring="accuracy",n_jobs=-1)
rf_search.fit(x_train,y_train)
rf_pred=rf_search.predict(x_test)

In [ ]:
xg_pipline=Pipeline([("perp",preprocess),("xg",XGBClassifier(subsample=0.8,colsample_bytree=0.8,random_state=0))])
param_grid={'xg__n_estimators':[100,150,200,250,300],"xg__learning_rate":[0.05,0.1,0.15,0.2,0.25],"xg__max_depth":[5,6,7,8,9,10]}
xg_search=GridSearchCV(xg_pipline,param_grid,cv=5,scoring="accuracy",n_jobs=-1)
xg_search.fit(x_train,y_train)
xg_pred=xg_search.predict(x_test)

In [46]:
def report(y_true,y_pred):
    return print(f"F1 score:{f1_score(y_true,y_pred)}\nRSME:{root_mean_squared_error(y_true,y_pred)}\nAccuracy:{accuracy_score(y_true,y_pred)}")
predictions= [("Logistic Regerssion",log_pred),("Decision Tree",dt_pred),("Random Forest",rf_pred),('SVM',svm_pred),("XGboost",xg_pred)]
for name, preds in predictions:
    print(f"{name} Report")
    report(y_test, preds)
# dont know why but the Rsme,r2score  and accuracy score are all the same

Logistic Regerssion Report
F1 score:0.7448275862068966
RSME:0.45592232730951643
Accuracy:0.7921348314606742
Decision Tree Report
F1 score:0.725925925925926
RSME:0.45592232730951643
Accuracy:0.7921348314606742
Random Forest Report
F1 score:0.75177304964539
RSME:0.4434289272060925
Accuracy:0.8033707865168539
SVM Report
F1 score:0.17721518987341772
RSME:0.6042917667202182
Accuracy:0.6348314606741573
XGboost Report
F1 score:0.7761194029850746
RSME:0.41053541362798
Accuracy:0.8314606741573034


In [47]:
print((log_pred == rf_pred).all(), (svm_pred == xg_pred).all())

False False
